# The coordinator: from a broad topic to subagent tasks

Run this notebook with this folder's `.venv` as the kernel. It uses the local Ollama model by default, so Ollama must be running. A run takes about a minute.

The notebook imports the real code from `research_coordinator/`, so nothing here is a separate copy.

## 1. Look at the hub

The graph is the hub. Paste this Mermaid text into [mermaid.live](https://mermaid.live) to see it drawn.

In [ ]:
from uuid import uuid4

from langgraph.types import Command

from research_coordinator.coordinator import build_coordinator, start_state

coordinator = build_coordinator()
print(coordinator.get_graph().draw_mermaid())

## 2. Give it a topic

Try a vague topic, then a specific one, and compare what happens. If the coordinator needs more information, the result contains `__interrupt__`: the graph has paused and saved its state under this `thread_id`.

In [ ]:
TOPIC = "AI in healthcare"

config = {"configurable": {"thread_id": str(uuid4())}}
result = coordinator.invoke(start_state(TOPIC), config)

result.get("__interrupt__", "No questions: the topic was clear enough.")

## 3. Answer its questions

`Command(resume=...)` wakes the paused graph and hands your answers to `interrupt()`. Leave an answer blank to skip it. The coordinator may ask a second round, up to the limit of 2.

In [ ]:
while "__interrupt__" in result:
    questions = result["__interrupt__"][0].value["questions"]
    answers = [input(question) for question in questions]
    result = coordinator.invoke(Command(resume=answers), config)

result["clarifications"]

## 4. Read the brief

This is the contract that later steps will plan from. Check `assumptions`: it lists what the coordinator decided without you.

In [ ]:
import json

print(json.dumps(result["brief"], indent=2))

## 5. Read the subagent tasks

Each task is everything one subagent will ever see. Read one `context` and ask yourself: could someone do this work with no other information?

In [ ]:
print(json.dumps(result["tasks"], indent=2))

## 6. See the hub's memory

The checkpointer kept a snapshot after every step. `next` shows which node was due to run, newest first. Notice `ask_user` appearing where the graph paused.

In [ ]:
for snapshot in coordinator.get_state_history(config):
    print(snapshot.next, "| rounds:", snapshot.values.get("clarification_rounds"))